Quantify drift with PSI and KS for features (e.g., log_amount, Time) and model scores (Test vs Stream1/2).

In [6]:
import src.preprocessing as prep
print(dir(prep))




['AmountTimeScaler', 'BaseEstimator', 'StandardScaler', 'TransformerMixin', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'np', 'pd', 'preprocess_features', 'split_x_y']


In [7]:
import pandas as pd
from pathlib import Path

DATA_PROCESSED = Path("../data/processed")

train   = pd.read_csv(DATA_PROCESSED / "batch1_train.csv")
test    = pd.read_csv(DATA_PROCESSED / "batch2_test.csv")
stream1 = pd.read_csv(DATA_PROCESSED / "batch3_stream.csv")
stream2 = pd.read_csv(DATA_PROCESSED / "batch4_stream.csv")

In [8]:
import numpy as np, pandas as pd
from pathlib import Path
from src.preprocessing import split_x_y, AmountTimeScaler
from src.utils import psi, ks_statistic
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

DATA_PROCESSED = Path("../data/processed")
test  = pd.read_csv(DATA_PROCESSED / "batch2_test.csv")
s1    = pd.read_csv(DATA_PROCESSED / "batch3_stream.csv")
s2    = pd.read_csv(DATA_PROCESSED / "batch4_stream.csv")


In [9]:
from src.preprocessing import split_x_y, preprocess_features

In [10]:
import numpy as np
import pandas as pd

def add_log_amount(df):
    df = df.copy()

    df["log_amount"] = np.log1p(df["amount"])

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )

    df["hour"] = df["timestamp"].dt.hour

    return df


# Create transformed datasets
test_f = add_log_amount(test)
s1_f   = add_log_amount(s1)
s2_f   = add_log_amount(s2)


# Compute PSI on numeric features only
for feat in ["log_amount", "hour"]:

    psi_s1 = psi(
        test_f[feat],
        s1_f[feat],
        bins=10
    )

    psi_s2 = psi(
        test_f[feat],
        s2_f[feat],
        bins=10
    )

    print(
        f"PSI({feat}) Test->S1={psi_s1:.3f}, "
        f"Test->S2={psi_s2:.3f}"
    )


PSI(log_amount) Test->S1=0.000, Test->S2=0.000
PSI(hour) Test->S1=0.001, Test->S2=0.001


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Train on processed features
X_tr, y_tr = split_x_y(train)
X_tr = preprocess_features(X_tr)

train_medians = X_tr.median(numeric_only=True)
X_tr = X_tr.fillna(train_medians)

pipe = Pipeline([
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

pipe.fit(X_tr, y_tr)


def scores(df):

    X, y = split_x_y(df)

    X = preprocess_features(X)

    X = X.reindex(
        columns=X_tr.columns,
        fill_value=0
    )

    X = X.fillna(train_medians)

    return pipe.predict_proba(X)[:,1], y


s_test, y_test = scores(test)
s_s1, y_s1     = scores(s1)
s_s2, y_s2     = scores(s2)

print("KS(score) Test vs S1:", ks_statistic(s_test, s_s1))
print("KS(score) Test vs S2:", ks_statistic(s_test, s_s2))

/Users/gyanvi/Downloads/fraud-detection-drift-aware/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


KS(score) Test vs S1: 0.0050080000000000124
KS(score) Test vs S2: 0.015936000000000006
